# Clustering socio-demográfico: flujo ordenado

Este cuaderno agrupa individuos en perfiles (clusters) usando sus variables socio-demográficas. Luego podremos usar el cluster como entrada para modelos posteriores.

Notas clave:
- WEIGHT_INDIV: es el peso de encuesta (factor de expansión). No se usa para entrenar KMeans/GMM, pero sí para perfilar y estimar porcentajes representativos por cluster (promedios y proporciones ponderadas).
- Orden de ejecución: 1) Cargar datos, 2) Clasificar columnas, 3) Preprocesamiento y prueba de K, 4) Gráficas de métricas, 5) Modelo final y perfilado, 6) (Opcional) Comparativa con GaussianMixture por BIC.


Agrupar 3337 individuos usando sólo columnas socio-demográficas. Más adelante añadiremos movilidad.

Pasos:
1. Clasificar columnas en: identificador, peso, numéricas, categóricas binarias/multiclase.
2. Preprocesar: imputación (mediana / modo), escalado numéricos, one-hot categóricas.
3. Probar KMeans para K=2..12 (inercia + silhouette) y elegir un K inicial (criterio combinación de codo + silhouette + interpretabilidad).
4. Ajustar modelo final, asignar cluster.
5. Perfilado ponderado (usar WEIGHT_INDIV) de cada cluster.
6. (Opcional) Comparar con GaussianMixture (BIC/AIC) para soft clusters.

Nota sobre WEIGHT_INDIV: es el factor de expansión (peso de encuesta). Usarlo para:
- Calcular proporciones representativas de la población.
- Calcular medias ponderadas. No usarlo dentro de KMeans (afecta escala); se aplica sólo al resumir.

Luego guardamos asignaciones para usar como feature en modelos posteriores.

Ejecuta las celdas en orden. Si falta alguna librería (scikit-learn), instalarla antes: pip install scikit-learn seaborn.


In [9]:
import numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import pandas as pd
import seaborn as sns

In [10]:
data = pd.read_csv('individuals_dataset_cleaned.csv')
data.head()

,ID,CODGEO,SEX,AGE,DIPLOMA,PRO_CAT,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,...,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD
0,10_2978,78092.0,0.0,41.0,0.0,4.0,2.0,1.0,0.0,0.0,...,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1856.206160,1.0
1,10_2980,75120.0,1.0,30.0,5.0,2.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1375.000372,1.0
2,10_2981,91326.0,1.0,38.0,5.0,2.0,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1231.812990,1.0
3,10_2982,91573.0,1.0,43.0,4.0,2.0,1.0,1.0,0.0,0.0,...,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,426.311616,1.0
4,10_2984,78073.0,0.0,39.0,5.0,2.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,843.194726,1.0


In [11]:
#listar colunas
data.columns

Index(['ID', 'CODGEO', 'SEX', 'AGE', 'DIPLOMA', 'PRO_CAT', 'NBPERS_HOUSE',
       'NB_10', 'NB_11_17', 'NB_18_24', 'NB_25_64', 'NB_65', 'PMR',
       'DRIVING_LICENCE', 'NB_CAR', 'TWO_WHEELER', 'BIKE', 'ELECT_SCOOTER',
       'NAVIGO_SUB', 'IMAGINER_SUB', 'OTHER_SUB_PT', 'BIKE_SUB', 'NSM_SUB',
       'WEIGHT_INDIV', 'GPS_RECORD'],
      dtype='object')

### Notas siguientes
- Ajusta K_FINAL tras revisar métricas + perfilado.
- Interpreta cada cluster: construir tabla cruzada de DIPLOMA y PRO_CAT por cluster.
- Guardar CSV con asignaciones para futuros modelos.
- Próximo paso: añadir variables de movilidad (cuando estén) y repetir.


In [12]:
# Alternativa: GaussianMixture con selección por BIC
from sklearn.mixture import GaussianMixture

# Asegurar transformaciones aplicadas con el preprocessor ya definido y ajustado en cada loop
X_prepared = preprocessor.fit_transform(X)

bic_list = []
for k in range(2, 11):
    gmm = GaussianMixture(n_components=k, covariance_type='full', n_init=5, random_state=42)
    gmm.fit(X_prepared)
    bic_list.append({'k': k, 'bic': gmm.bic(X_prepared), 'aic': gmm.aic(X_prepared)})

bic_df = pd.DataFrame(bic_list)
bic_df.sort_values('bic').reset_index(drop=True)


,k,bic,aic
0,10,-356266.045989,-394770.746278
1,9,-338502.554911,-373156.173888
2,8,-330671.654019,-361474.191685
3,7,-325308.873858,-352260.330212
4,6,-322390.306201,-345490.681243
5,5,-271369.267897,-290618.561628
6,4,-240177.284816,-255575.497235
7,3,-208524.688129,-220071.819236
8,2,-118594.801709,-126290.851505


In [13]:
# Elegir K (ajusta este valor tras ver la gráfica / tabla)
K_FINAL = 6  # <-- cambia manualmente si ves mejor opción
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

final_pipe = Pipeline([
    ('prep', preprocessor),
    ('clust', KMeans(n_clusters=K_FINAL, n_init=30, random_state=42))
])
final_labels = final_pipe.fit_predict(X)

data['CLUSTER_KMEANS'] = final_labels

# Guardar pipeline para reutilizar (opcional)
import joblib
joblib.dump(final_pipe, 'kmeans_cluster_pipeline.pkl')

# Perfilado ponderado
import numpy as np

def weighted_profile(df, cluster_col, weight_col):
    w_all = df[weight_col].fillna(0)
    g = df.groupby(cluster_col, dropna=False)
    out = []
    for c, sub in g:
        w_sub = sub[weight_col].fillna(0)
        tot_w = w_sub.sum()
        tot_w = float(tot_w) if tot_w > 0 else 1.0  # evitar división por 0
        row = {
            'cluster': int(c),
            'n_individuos': int(len(sub)),
            'peso_total': float(w_sub.sum()),
            'pct_peso': float(w_sub.sum() / max(w_all.sum(), 1.0))
        }
        # medias numéricas
        for col in numeric_cols:
            num = (sub[col].fillna(sub[col].median()) * w_sub).sum()
            den = w_sub.sum() if w_sub.sum() > 0 else len(sub)
            row[f'{col}_mean'] = float(num / den)
        # proporciones binarias (valor 1)
        for col in binary_cols:
            num = (sub[col].fillna(0) * w_sub).sum()
            den = w_sub.sum() if w_sub.sum() > 0 else len(sub)
            row[f'{col}_pct1'] = float(num / den)
        out.append(row)
    return pd.DataFrame(out)

profile_df = weighted_profile(data, 'CLUSTER_KMEANS', weight_col)
profile_df.sort_values('pct_peso', ascending=False)


,cluster,n_individuos,peso_total,pct_peso,AGE_mean,NBPERS_HOUSE_mean,NB_10_mean,NB_11_17_mean,NB_18_24_mean,NB_25_64_mean,...,DRIVING_LICENCE_pct1,TWO_WHEELER_pct1,BIKE_pct1,ELECT_SCOOTER_pct1,NAVIGO_SUB_pct1,IMAGINER_SUB_pct1,OTHER_SUB_PT_pct1,BIKE_SUB_pct1,NSM_SUB_pct1,GPS_RECORD_pct1
1,1,747,2.228174e+06,0.246368,45.000747,2.708508,0.260492,0.412968,0.164312,1.851787,...,0.868294,0.133827,0.745047,0.125225,0.501096,0.047433,0.044480,0.070630,0.022725,0.997139
4,4,1151,2.031500e+06,0.224622,43.290989,1.348535,0.108725,0.094147,0.135338,0.567061,...,0.753349,0.042577,0.392513,0.074576,0.667616,0.090872,0.068427,0.118192,0.012856,0.993421
5,5,620,1.638384e+06,0.181155,40.918206,3.630957,0.484795,0.986133,0.319224,1.798876,...,0.819003,0.166671,3.427579,0.192446,0.385412,0.116370,0.045994,0.108336,0.027460,0.992842
0,0,258,1.100840e+06,0.121719,65.408569,2.269222,0.122426,0.057967,0.115850,0.459537,...,0.914971,0.095749,0.880986,0.037790,0.430982,0.018185,0.111848,0.040386,0.023191,0.997624
2,2,289,1.064397e+06,0.117690,40.472782,4.579387,2.533972,0.332875,0.141389,1.452457,...,0.797561,0.065109,1.150368,0.113711,0.490395,0.065274,0.066100,0.052915,0.024748,0.991696
3,3,272,9.807912e+05,0.108446,27.993260,3.849106,0.148610,0.411419,1.707519,1.557607,...,0.500068,0.147184,0.925637,0.205752,0.427339,0.436464,0.019807,0.110216,0.009399,1.000000


In [16]:
# Clasificación de columnas y validaciones rápidas
id_col = 'ID'
weight_col = 'WEIGHT_INDIV'

# Columnas del dataset
cols = data.columns.tolist()

# Binarias (0/1)
binary_cols = ['SEX','PMR','DRIVING_LICENCE','TWO_WHEELER','BIKE','ELECT_SCOOTER',
               'NAVIGO_SUB','IMAGINER_SUB','OTHER_SUB_PT','BIKE_SUB','NSM_SUB','GPS_RECORD']

# Numéricas (contadores / edad)
numeric_cols = ['AGE','NBPERS_HOUSE','NB_10','NB_11_17','NB_18_24','NB_25_64','NB_65','NB_CAR']

# Categóricas multiclase (tratadas con OneHot)
cat_cols = ['DIPLOMA','PRO_CAT']

# Chequeos
print('Resumen columnas (primeras 5 filas):')
display(data.head())
print('\nRevisión tipos y nulos:')
for c in numeric_cols + binary_cols:
    print(f"{c:15s} dtype={data[c].dtype}  n_missing={data[c].isna().sum()}")

print('\nCategorías (cardinalidad y ejemplos):')
for c in cat_cols:
    uniq = data[c].dropna().unique()
    print(f"{c:15s} n_unique={data[c].nunique()}  ejemplos={uniq[:6]}")

# Comprobación de exclusiones
used = set([id_col, weight_col] + binary_cols + numeric_cols + cat_cols)
unused = [c for c in cols if c not in used]
print('\nColumnas no usadas por ahora:', unused)


Resumen columnas (primeras 5 filas):


,ID,CODGEO,SEX,AGE,DIPLOMA,PRO_CAT,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,...,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD,CLUSTER_KMEANS
0,10_2978,78092.0,0.0,41.0,0.0,4.0,2.0,1.0,0.0,0.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1856.206160,1.0,1
1,10_2980,75120.0,1.0,30.0,5.0,2.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1375.000372,1.0,4
2,10_2981,91326.0,1.0,38.0,5.0,2.0,2.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1231.812990,1.0,1
3,10_2982,91573.0,1.0,43.0,4.0,2.0,1.0,1.0,0.0,0.0,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,426.311616,1.0,4
4,10_2984,78073.0,0.0,39.0,5.0,2.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,843.194726,1.0,4



Revisión tipos y nulos:
AGE             dtype=float64  n_missing=0
NBPERS_HOUSE    dtype=float64  n_missing=0
NB_10           dtype=float64  n_missing=0
NB_11_17        dtype=float64  n_missing=0
NB_18_24        dtype=float64  n_missing=0
NB_25_64        dtype=float64  n_missing=0
NB_65           dtype=float64  n_missing=0
NB_CAR          dtype=float64  n_missing=0
SEX             dtype=float64  n_missing=0
PMR             dtype=float64  n_missing=0
DRIVING_LICENCE dtype=float64  n_missing=0
TWO_WHEELER     dtype=float64  n_missing=0
BIKE            dtype=float64  n_missing=0
ELECT_SCOOTER   dtype=float64  n_missing=0
NAVIGO_SUB      dtype=float64  n_missing=0
IMAGINER_SUB    dtype=float64  n_missing=0
OTHER_SUB_PT    dtype=float64  n_missing=0
BIKE_SUB        dtype=float64  n_missing=0
NSM_SUB         dtype=float64  n_missing=0
GPS_RECORD      dtype=float64  n_missing=0

Categorías (cardinalidad y ejemplos):
DIPLOMA         n_unique=6  ejemplos=[0. 5. 4. 1. 3. 2.]
PRO_CAT         n_u

In [17]:
# Preprocesamiento y prueba de distintos K
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Pipelines de transformación
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # No escalamos binarios (0/1) para mantener interpretabilidad parcial
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('bin', binary_pipeline, binary_cols),
    ('cat', categorical_pipeline, cat_cols)
], remainder='drop')

X = data[numeric_cols + binary_cols + cat_cols]

k_list = list(range(2, 13))
results = []

for k in k_list:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clust', KMeans(n_clusters=k, n_init=30, random_state=42))
    ])
    labels = pipe.fit_predict(X)
    X_tr = pipe.named_steps['prep'].transform(X)
    sil = silhouette_score(X_tr, labels, metric='euclidean')
    inertia = pipe.named_steps['clust'].inertia_
    results.append({'k': k, 'silhouette': sil, 'inertia': inertia})

res_df = pd.DataFrame(results)
res_df.sort_values('silhouette', ascending=False).reset_index(drop=True)


,k,silhouette,inertia
0,3,0.196332,31708.138084
1,2,0.195274,35059.315946
2,4,0.178302,29108.089983
3,5,0.174846,27376.755415
4,6,0.137950,25895.647252
5,7,0.136002,24705.027644
6,9,0.123850,22884.270361
7,11,0.121809,21543.468538
8,12,0.121290,21026.649424
9,10,0.118496,22161.304038


<h2> Visualizacion

<h3> Visualizacion de Clusters

In [18]:
#visualizar la clusterizacion con k=3 en dos dimension

<h3> Visualizacion de la metrica de error</h3>

In [ ]:
# Visualizar métricas para elegir K
assert 'res_df' in globals(), 'Ejecuta primero la celda que calcula res_df'
fig, ax1 = plt.subplots(figsize=(8,4))
color = 'tab:blue'
ax1.set_xlabel('k')
ax1.set_ylabel('Inertia', color=color)
ax1.plot(res_df['k'], res_df['inertia'], marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette', color=color)
ax2.plot(res_df['k'], res_df['silhouette'], marker='s', color=color)
ax2.tick_params(axis='y', labelcolor=color)
plt.title('Inertia vs Silhouette')
plt.show()
res_df.sort_values('silhouette', ascending=False).head()


# Extra

In [ ]:
# Preprocesamiento y prueba de distintos K
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Selección de features para clustering (excluye id, peso)
features_binary = binary_cols  # ya 0/1, igual podemos tratarlas como numéricas sin escalado extra
features_numeric = numeric_cols
features_categorical = cat_cols

# Pipelines de transformación
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # no escalamos binarios (0/1) para mantener interpretabilidad parcial
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, features_numeric),
    ('bin', binary_pipeline, features_binary),
    ('cat', categorical_pipeline, features_categorical)
], remainder='drop')

X = data[features_numeric + features_binary + features_categorical]

k_list = list(range(2, 13))
results = []
X_trans = None
for k in k_list:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clust', KMeans(n_clusters=k, n_init=20, random_state=42))
    ])
    labels = pipe.fit_predict(X)
    # Guardar primero para k elegido si queremos reproducir
    if k == 2:
        X_trans = pipe.named_steps['prep'].transform(X)  # guardar ejemplo
    sil = silhouette_score(pipe.named_steps['prep'].transform(X), labels)
    inertia = pipe.named_steps['clust'].inertia_
    results.append({'k': k, 'silhouette': sil, 'inertia': inertia})

res_df = pd.DataFrame(results)
res_df
